# HVI 빅캠 6회차 - 고매칭 버전 (PNU 매칭률 높을 때)

## 실행 방법
- `month_str`만 바꿔서 월별로 각각 실행 후 반출
- 로컬에서 3개월 평균 산출

## 주요 변경사항
- bnd shapefile 기반 행정동 매핑 + 통폐합(상일1동→상일동, 상일2동→강일동) 반영
- 집적도: DJAREA > 0 AND SEDECNT notna 동시 필터링 (분자·분모 동일 집합)
- 집적도 없는 동: 2축(저가비율+노후도) 평균으로 HVI 산출
- B068 + 건축물대장에만 있는 건물 통합 (PNU 중복 제외)

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.stats import rankdata

month_str = '202208'
# month_str = '202207'
# month_str = '202206'

ref_year = 2022
LOW_Q = 0.2
old_threshold = 30
min_building = 5
MATCH_THRESHOLD = 0.7

dong_rename = {'상일1동': '상일동', '상일2동': '강일동'}
code_rename = {'11250760': '11250760', '11250770': '11250750'}

# 1. 데이터 로드

In [ ]:
def load_csv(txt_dir):
    df = pd.read_csv('E:/B068. 서울시 연립 다세대 임대 예측시세/2. 파일데이터/' + txt_dir, sep='|', dtype=str, engine='python')
    df.columns = [str(c).strip().strip('`') for c in df.columns]
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.strip('`')
    return df

def load_month(month_str):
    return (
        load_csv(f'1. 주택기본정보/jtbasicinfo_{month_str}.txt'),
        load_csv(f'3. 전세임대 예측시세/depo_hosiseinfo_{month_str}.txt'),
        load_csv(f'5. 월세임대 예측시세/rent_hosiseinfo_{month_str}.txt')
    )

df_주택기본정보, df_전세예측, df_월세예측 = load_month(month_str)
df_bldg = pd.read_csv('건축물대장_빅캠반입용.csv', dtype={'hdong_code': str, 'PNU': str})

print(f'[{month_str}] B068 주택기본정보: {len(df_주택기본정보):,}행')
print(f'건축물대장: {len(df_bldg):,}행')

In [ ]:
# B068 결측 현황 확인
import pandas as pd

# ── 주택기본정보 ──────────────────────────────
print("=" * 50)
print("주택기본정보 결측 현황")
print("=" * 50)
print(f"총 행 수: {len(df_주택기본정보):,}")
missing = df_주택기본정보.isnull().sum()
missing_pct = (missing / len(df_주택기본정보) * 100).round(2)
summary = pd.DataFrame({'결측수': missing, '결측률(%)': missing_pct})
print(summary[summary['결측수'] > 0].sort_values('결측수', ascending=False))

# ── 전세예측 ──────────────────────────────────
print("\n" + "=" * 50)
print("전세예측 결측 현황")
print("=" * 50)
print(f"총 행 수: {len(df_전세예측):,}")
missing = df_전세예측.isnull().sum()
missing_pct = (missing / len(df_전세예측) * 100).round(2)
summary = pd.DataFrame({'결측수': missing, '결측률(%)': missing_pct})
print(summary[summary['결측수'] > 0].sort_values('결측수', ascending=False))

# ── 월세예측 ──────────────────────────────────
print("\n" + "=" * 50)
print("월세예측 결측 현황")
print("=" * 50)
print(f"총 행 수: {len(df_월세예측):,}")
missing = df_월세예측.isnull().sum()
missing_pct = (missing / len(df_월세예측) * 100).round(2)
summary = pd.DataFrame({'결측수': missing, '결측률(%)': missing_pct})
print(summary[summary['결측수'] > 0].sort_values('결측수', ascending=False))

# 2. B068 행정동 매핑 (bnd shapefile + 통폐합)

In [ ]:
gdf_hdong = gpd.read_file('bnd_dong_11_2025_2Q.shp')
gdf_hdong = gdf_hdong.to_crs(epsg=4326)
gdf_hdong = gdf_hdong.rename(columns={'ADM_CD': 'hdong_code', 'ADM_NM': 'hdong_name'})

df_주택기본정보['LNG'] = pd.to_numeric(df_주택기본정보['LNG'], errors='coerce')
df_주택기본정보['LAT'] = pd.to_numeric(df_주택기본정보['LAT'], errors='coerce')

gdf_bldg_b068 = gpd.GeoDataFrame(
    df_주택기본정보,
    geometry=gpd.points_from_xy(df_주택기본정보['LNG'], df_주택기본정보['LAT']),
    crs='EPSG:4326'
)
gdf_joined = gpd.sjoin(gdf_bldg_b068, gdf_hdong[['hdong_code', 'hdong_name', 'geometry']], how='left', predicate='within')

gdf_joined['hdong_code'] = gdf_joined['hdong_code'].replace(code_rename)
gdf_joined['hdong_name'] = gdf_joined['hdong_name'].replace(dong_rename)

print(f'행정동 매핑 완료. 결측: {gdf_joined["hdong_code"].isna().mean():.1%}')

df_주택기본정보 = pd.DataFrame(gdf_joined.drop(columns='geometry'))
df_전세예측 = df_전세예측.merge(
    df_주택기본정보[['PKCODE1', 'hdong_code', 'hdong_name']].drop_duplicates('PKCODE1'),
    on='PKCODE1', how='left'
)
df_월세예측 = df_월세예측.merge(
    df_주택기본정보[['PKCODE1', 'hdong_code', 'hdong_name']].drop_duplicates('PKCODE1'),
    on='PKCODE1', how='left'
)

# 3. PNU 매칭률 확인

In [ ]:
df_주택기본정보['PNU'] = df_주택기본정보['PNU'].astype(str).str.strip()
b068_pnu = set(df_주택기본정보['PNU'].dropna().unique())
bldg_pnu = set(df_bldg['PNU'].dropna().unique())

matched = b068_pnu & bldg_pnu
match_rate = len(matched) / len(b068_pnu)

print(f'B068 고유 건물 수: {len(b068_pnu):,}')
print(f'건축물대장 고유 건물 수: {len(bldg_pnu):,}')
print(f'매칭된 건물 수: {len(matched):,}')
print(f'매칭률 (B068 기준): {match_rate:.1%}')

if match_rate >= MATCH_THRESHOLD:
    print(f'\n✅ 고매칭 버전 진행')
else:
    print(f'\n⚠️ 저매칭 버전 사용 권장')

# 4. 통합 데이터프레임 구성 (B068 + 건축물대장에만 있는 건물)

In [ ]:
df_주택기본정보['DJAREA'] = pd.to_numeric(df_주택기본정보['DJAREA'], errors='coerce')
df_주택기본정보['SEDECNT'] = pd.to_numeric(df_주택기본정보['SEDECNT'], errors='coerce')

df_b068_dedup = df_주택기본정보[['PNU', 'PKCODE1', 'SYDATE', 'DJAREA', 'SEDECNT', 'hdong_code', 'hdong_name']].drop_duplicates('PNU').copy()
df_b068_dedup['source'] = 'B068'

df_bldg['DJAREA'] = pd.to_numeric(df_bldg['total_area'], errors='coerce')
df_bldg['SEDECNT'] = pd.to_numeric(df_bldg['total_units'], errors='coerce')
df_bldg_only = df_bldg[~df_bldg['PNU'].isin(b068_pnu)].copy()
df_bldg_only['PKCODE1'] = np.nan
df_bldg_only['source'] = 'bldg_only'

common_cols = ['PNU', 'PKCODE1', 'SYDATE', 'DJAREA', 'SEDECNT', 'hdong_code', 'hdong_name', 'source']
df_통합 = pd.concat([
    df_b068_dedup[common_cols],
    df_bldg_only[common_cols]
], ignore_index=True)

print(f'B068 건물: {len(df_b068_dedup):,}')
print(f'건축물대장에만 있는 건물: {len(df_bldg_only):,}')
print(f'통합 건물 수: {len(df_통합):,}')

# 5. 저가비율 (B068 기반)

In [ ]:
dong_code_col = 'hdong_code'
price_col_depo = 'DEPO_PRED'
price_col_rent = 'PRED_RENT'

df_전세예측[price_col_depo] = pd.to_numeric(df_전세예측[price_col_depo], errors='coerce')
df_월세예측[price_col_rent] = pd.to_numeric(df_월세예측[price_col_rent], errors='coerce')

def prepare_price_ratio(df, price_col, dong_code_col, label):
    tmp = df[df[price_col].notna()].copy()
    q = tmp[price_col].quantile(LOW_Q)
    tmp[f'is_low_{label}'] = (tmp[price_col] <= q).astype(int)
    agg = (tmp.groupby(dong_code_col, dropna=False)
             .agg(n_obs=(price_col, 'size'), low_ratio=(f'is_low_{label}', 'mean'))
             .reset_index())
    agg = agg.rename(columns={'n_obs': f'n_{label}', 'low_ratio': f'low_ratio_{label}'})
    return agg, q

depo_agg, depo_q = prepare_price_ratio(df_전세예측, price_col_depo, dong_code_col, label='depo')
rent_agg, rent_q = prepare_price_ratio(df_월세예측, price_col_rent, dong_code_col, label='rent')

print(f'전세 저가 기준: {depo_q:,.0f}만원')
print(f'월세 저가 기준: {rent_q:,.0f}만원')

# 6. 노후도 (통합 데이터 기반)

In [ ]:
df_통합['SYDATE_parsed'] = pd.to_datetime(df_통합['SYDATE'], format='%Y%m%d', errors='coerce')
df_통합['build_year_final'] = df_통합['SYDATE_parsed'].dt.year

df_old = df_통합[df_통합['build_year_final'].notna()].copy()
df_old['building_age'] = ref_year - df_old['build_year_final'].astype(int)
df_old['is_old'] = (df_old['building_age'] >= old_threshold).astype(int)

old_agg = (df_old.groupby(dong_code_col, dropna=False)
           .agg(n_building=('building_age', 'size'), old_ratio=('is_old', 'mean'))
           .reset_index())

print(f'노후도 집계 행정동 수: {len(old_agg)}')

# 7. 집적도 (통합 데이터 기반)

DJAREA > 0 AND SEDECNT notna 동시 필터링 → 분자·분모 동일 집합 보장.

In [ ]:
df_density = df_통합[
    (df_통합['DJAREA'] > 0) &
    df_통합['DJAREA'].notna() &
    df_통합['SEDECNT'].notna()
].copy()

print(f'집적도 계산 대상: {len(df_density):,}건 / 전체 {len(df_통합):,}건 ({1 - len(df_density)/len(df_통합):.1%} 제외)')

density_agg = (df_density.groupby(dong_code_col, dropna=False)
               .agg(
                   n_building_density=(dong_code_col, 'size'),
                   total_units=('SEDECNT', 'sum'),
                   total_area=('DJAREA', 'sum')
               )
               .reset_index())

density_agg['unit_density'] = density_agg['total_units'] / density_agg['total_area']
print(f'집적도 집계 행정동 수: {len(density_agg)}')

# 8. HVI 통합 merge

In [ ]:
hvi = pd.merge(depo_agg, rent_agg, on='hdong_code', how='outer')
hvi['n_rent'] = hvi['n_rent'].fillna(0)

def weighted_low_ratio(row):
    n_depo = row['n_depo']
    n_rent = row['n_rent']
    total = n_depo + n_rent
    if total == 0:
        return np.nan
    val = 0
    if pd.notna(row['low_ratio_depo']):
        val += n_depo * row['low_ratio_depo']
    if pd.notna(row['low_ratio_rent']):
        val += n_rent * row['low_ratio_rent']
    return val / total

hvi['low_ratio_combined'] = hvi.apply(weighted_low_ratio, axis=1)
hvi = pd.merge(hvi, old_agg, on='hdong_code', how='outer')
hvi = pd.merge(hvi, density_agg, on='hdong_code', how='outer')

hdong_name_map = pd.concat([
    df_주택기본정보[['hdong_code', 'hdong_name']],
    df_bldg[['hdong_code', 'hdong_name']]
]).drop_duplicates('hdong_code')
hvi = hvi.merge(hdong_name_map, on='hdong_code', how='left')

print(f'merge 후 행정동 수: {len(hvi)}')
print(hvi[['low_ratio_combined', 'old_ratio', 'unit_density']].isna().sum())

# 9. B068 누락 행정동 저가비율 → 자치구 평균 대체

> **[수정 4/20]** 집적도 결측도 동일하게 자치구 평균 대체로 처리. 두 축 모두 결측 원인이 B068 커버리지/집합건물 구조상 데이터 품질 문제로 판단 (실제 건물 존재 확인). ANOVA 검정으로 타당성 확인 (저가비율 F=8.73 p<0.001, 집적도 F=2.06 p=0.003).


In [ ]:
hvi['gu_code'] = hvi['hdong_code'].astype(str).str[:4]

# 저가비율 구 평균 대체
gu_mean_low = hvi.groupby('gu_code')['low_ratio_combined'].mean()
print(f'저가비율 결측 행정동: {hvi["low_ratio_combined"].isna().sum()}개')
hvi['low_ratio_imputed'] = hvi['low_ratio_combined'].isna()
hvi['low_ratio_combined'] = hvi.apply(
    lambda row: gu_mean_low.get(row['gu_code'], np.nan)
    if pd.isna(row['low_ratio_combined']) else row['low_ratio_combined'],
    axis=1
)
print(f'저가비율 대체 후 결측: {hvi["low_ratio_combined"].isna().sum()}개')

# [수정 4/20] 집적도 구 평균 대체 추가
# 근거: 집합건물 총괄표제부 구조상 세대수·면적 미기재 → 데이터 품질 문제
# ANOVA: 구 내 분산 88% → 구 평균 대체 안정적 (저가비율보다 오히려 더 균일)
gu_mean_density = hvi.groupby('gu_code')['unit_density'].mean()
print(f'\n집적도 결측 행정동: {hvi["unit_density"].isna().sum()}개')
hvi['density_imputed'] = hvi['unit_density'].isna()
hvi['unit_density'] = hvi.apply(
    lambda row: gu_mean_density.get(row['gu_code'], np.nan)
    if pd.isna(row['unit_density']) else row['unit_density'],
    axis=1
)
print(f'집적도 대체 후 결측: {hvi["unit_density"].isna().sum()}개')


# 10. 최소 표본 필터링 및 점수 산출

> **[수정 4/20]** 가중치 변경: 균등 1/3 → `저가:노후도:집적도 = 0.3:0.4:0.3` (노후도는 전 행정동 계산 가능한 가장 신뢰도 높은 축). 축 결측 시 해당 가중치 제외 후 정규화.
> **[수정 4/20]** 집적도 결측 동도 구 평균 대체로 3축 모두 산출 (n_axes 항상 3).
> **[수정 4/20]** `imputation_flag`, `reliability` 컬럼 추가.


In [ ]:
hvi['n_total'] = hvi['n_depo'].fillna(0) + hvi['n_rent'].fillna(0)

hvi_filtered = hvi[
    hvi['low_ratio_combined'].notna() &
    hvi['old_ratio'].notna() &
    (hvi['n_building'] >= min_building)
].copy()

print(f'필터링 후: {len(hvi_filtered)}개 동')
print(f'  집적도 실측 동: {(~hvi_filtered["density_imputed"]).sum()}개')
print(f'  집적도 보간 동: {hvi_filtered["density_imputed"].sum()}개')

def percentile_score(series):
    return (rankdata(series, method='average') - 1) / (len(series) - 1) * 100

hvi_filtered['score_low'] = percentile_score(hvi_filtered['low_ratio_combined'])
hvi_filtered['score_old'] = percentile_score(hvi_filtered['old_ratio'])
hvi_filtered['score_density'] = percentile_score(hvi_filtered['unit_density'])  # [수정 4/20] 전체 산출

# [수정 4/20] 가중치 기반 HVI 산출 (저가:노후도:집적도 = 0.3:0.4:0.3)
# 축 결측 시 해당 가중치 제외 후 정규화 (현재는 전 동 3축 산출되므로 항상 분모=1.0)
W_LOW, W_OLD, W_DENSITY = 0.3, 0.4, 0.3

def calc_hvi(row):
    scores, weights = {}, {}
    if pd.notna(row['score_low']):
        scores['low'] = row['score_low']; weights['low'] = W_LOW
    if pd.notna(row['score_old']):
        scores['old'] = row['score_old']; weights['old'] = W_OLD
    if pd.notna(row['score_density']):
        scores['density'] = row['score_density']; weights['density'] = W_DENSITY
    if not scores:
        return np.nan
    total_w = sum(weights.values())
    return sum(scores[k] * weights[k] for k in scores) / total_w

hvi_filtered['HVI_score'] = hvi_filtered.apply(calc_hvi, axis=1)

hvi_filtered['HVI_index'] = (
    (hvi_filtered['HVI_score'] - hvi_filtered['HVI_score'].min()) /
    (hvi_filtered['HVI_score'].max() - hvi_filtered['HVI_score'].min()) * 100
).round().astype(int)

hvi_filtered['HVI_rank'] = hvi_filtered['HVI_score'].rank(ascending=False, method='min')
hvi_filtered['HVI_grade'] = pd.qcut(
    hvi_filtered['HVI_score'], q=5,
    labels=['매우낮음', '낮음', '보통', '높음', '매우높음']
)
hvi_filtered['n_axes'] = 3  # [수정 4/20] 집적도 보간으로 항상 3축

# [수정 4/20] 신뢰도 컬럼 추가
# imputation_flag: 보간된 축 표시
def get_imputation_flag(row):
    imputed = []
    if row['low_ratio_imputed']: imputed.append('low_ratio')
    if row['density_imputed']: imputed.append('density')
    return '|'.join(imputed) if imputed else 'none'

hvi_filtered['imputation_flag'] = hvi_filtered.apply(get_imputation_flag, axis=1)

# reliability: 보간 축 수 기반 신뢰도 등급
def get_reliability(row):
    n_imputed = row['low_ratio_imputed'] + row['density_imputed']
    if n_imputed == 0: return 'high'
    if n_imputed == 1: return 'medium'
    return 'low'

hvi_filtered['reliability'] = hvi_filtered.apply(get_reliability, axis=1)
print('\n신뢰도 분포:')
print(hvi_filtered['reliability'].value_counts())


# 11. Sanity Check

In [ ]:
print('=== HVI 상위 20개 동 ===')
print(hvi_filtered.sort_values('HVI_rank').head(20)[
    ['hdong_name', 'old_ratio', 'low_ratio_combined', 'unit_density', 'HVI_score', 'HVI_grade', 'n_axes']
].to_string())

print('\n=== HVI_score 분포 ===')
print(hvi_filtered['HVI_score'].describe())

print('\n=== 등급별 동 수 ===')
print(hvi_filtered['HVI_grade'].value_counts())

print(f'\n=== 저가비율 자치구 평균 대체: {hvi_filtered["low_ratio_imputed"].sum()}개 동 ===')
print(hvi_filtered[hvi_filtered['low_ratio_imputed']][['hdong_name', 'gu_code', 'low_ratio_combined']].to_string())

# 12. 최종 저장

In [ ]:
export_cols = [
    'hdong_code', 'hdong_name',
    'low_ratio_combined', 'low_ratio_imputed',
    'old_ratio',
    'unit_density', 'density_imputed',  # [수정 4/20] density_imputed 추가
    'score_low', 'score_old', 'score_density',
    'HVI_score', 'HVI_index', 'HVI_rank', 'HVI_grade', 'n_axes',
    'imputation_flag', 'reliability'    # [수정 4/20] 신뢰도 컬럼 추가
]

hvi_filtered[export_cols].to_csv(f'HVI_bnd_dong_{month_str}_high_match.csv', index=False, encoding='utf-8-sig')
print(f'저장 완료: {len(hvi_filtered)}개 행정동 → HVI_bnd_dong_{month_str}_high_match.csv')
